In [5]:
import os
import pandas as pd

# Path configuration relative to the notebooks folder
RAW_DIR = os.path.join("..", "data", "raw_source")

# The standardized filenames you created
target_files = [
    "qcommerce_orders.csv",
    "qcommerce_order_items.csv",
    "qcommerce_customers.csv",
    "qcommerce_products.csv",
    "qcommerce_inventory.csv",
    "qcommerce_feedback.csv",
    "qcommerce_marketing.csv"
]

def audit_standardized_datasets():
    print(f"[#] Auditing files located in: {os.path.abspath(RAW_DIR)}\n")
    
    for filename in target_files:
        full_path = os.path.join(RAW_DIR, filename)
        
        if not os.path.exists(full_path):
            print(f"[!] Warning: Expected file not found -> {filename}")
            continue
            
        # Inspect a chunk to assess schema shape
        df = pd.read_csv(full_path, nrows=5)
        
        print("=" * 60)
        print(f"🎯 File Profile: {filename}")
        print(f"📈 Total columns: {len(df.columns)}")
        print("=" * 60)
        
        meta_df = pd.DataFrame({
            "Column Name": df.columns,
            "Inferred DataType": df.dtypes.values,
            "Sample Value": df.iloc[0].values
        })
        
        print(meta_df.to_string(index=False))
        print("\n")

audit_standardized_datasets()

[#] Auditing files located in: c:\Users\HP\Documents\Qcommerce\data\raw_source

🎯 File Profile: qcommerce_orders.csv
📈 Total columns: 10
           Column Name Inferred DataType        Sample Value
              order_id             int64          1961864118
           customer_id             int64            30065862
            order_date               str 2024-07-17 08:34:01
promised_delivery_time               str 2024-07-17 08:52:01
  actual_delivery_time               str 2024-07-17 08:47:01
       delivery_status               str             On Time
           order_total           float64             3197.07
        payment_method               str                Cash
   delivery_partner_id             int64               63230
              store_id             int64                4771


🎯 File Profile: qcommerce_order_items.csv
📈 Total columns: 4
Column Name Inferred DataType  Sample Value
   order_id             int64  1.961864e+09
 product_id             int64  6.426120e+

In [6]:
import os
import re
import pandas as pd

RAW_DIR = os.path.join("..", "data", "raw_source")

target_files = {
    "orders": "qcommerce_orders.csv",
    "order_items": "qcommerce_order_items.csv",
    "customers": "qcommerce_customers.csv",
    "products": "qcommerce_products.csv",
    "inventory": "qcommerce_inventory.csv",
    "feedback": "qcommerce_feedback.csv",
    "marketing": "qcommerce_marketing.csv"
}

def clean_diagnostics():
    print("=" * 70)
    print("🔍 RUNNING CRITICAL SYSTEM DIAGNOSTICS & DATA SANITY VERIFICATION")
    print("=" * 70)
    
    # -------------------------------------------------------------
    # DIAGNOSTIC 1: RAW DISK FILE LINE COUNT VERIFICATION
    # -------------------------------------------------------------
    print("\n📊 [1/3] VERIFYING RAW DISK LINE COUNTS (Checking for Truncation)...")
    disk_counts = {}
    for key, filename in target_files.items():
        path = os.path.join(RAW_DIR, filename)
        if os.path.exists(path):
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                lines = sum(1 for line in f)
            disk_counts[key] = lines
            print(f"  • {filename:<30} -> Raw Lines on Disk: {lines:,}")
        else:
            print(f"  • [!] FILE MISSING: {filename}")
            disk_counts[key] = 0

    # -------------------------------------------------------------
    # DIAGNOSTIC 2: FACT RATIO VALIDATION
    # -------------------------------------------------------------
    if disk_counts.get("orders") and disk_counts.get("order_items"):
        # Subtracting 1 for headers
        num_orders = disk_counts["orders"] - 1
        num_items = disk_counts["order_items"] - 1
        ratio = num_items / num_orders if num_orders > 0 else 0
        print(f"\n📈 Ratio Check: order_items to orders ratio is {ratio:.2f} items/order")
        if ratio < 1.0:
            print("  ⚠️ RED FLAG: Abnormally low items-to-orders ratio! Items table might be truncated.")
        else:
            print("  ✅ Pass: Items-to-orders ratio looks realistic.")

    # -------------------------------------------------------------
    # DIAGNOSTIC 3: INVENTORY PATTERN MASK INSPECTION ( mixed format check )
    # -------------------------------------------------------------
    print("\n🧩 [2/3] INVESTIGATING INVENTORY DATE PATTERNS (Mixed Format Check)...")
    inv_path = os.path.join(RAW_DIR, target_files["inventory"])
    if os.path.exists(inv_path):
        inv_df = pd.read_csv(inv_path)
        
        # Classify string structures using character maps
        patterns = inv_df['date'].astype(str).apply(lambda x: re.sub(r'\d', 'X', x)).value_counts()
        for pattern, count in patterns.items():
            print(f"  • Found Pattern Mapped Format: '{pattern}' -> Rows: {count:,}")
            
        # Parse cleanly with forced dayfirst constraint to fix the string sort bug
        inv_df['parsed_date'] = pd.to_datetime(inv_df['date'], dayfirst=True, errors='coerce')
        nat_count = inv_df['parsed_date'].isna().sum()
        if nat_count > 0:
            print(f"  ⚠️ ALERT: {nat_count} rows failed datetime conversion in inventory!")
            
        print(f"  • Actual Chronological Min Date: {inv_df['parsed_date'].min()}")
        print(f"  • Actual Chronological Max Date: {inv_df['parsed_date'].max()}")
    else:
        print("  • Skipped: Inventory file not present.")

    # -------------------------------------------------------------
    # DIAGNOSTIC 4: GLOBAL CHRONOLOGICAL ALIGNMENT
    # -------------------------------------------------------------
    print("\n🗓️ [3/3] EXTRACTING DEFINITIVE TIMELINES FOR TIMELINE STRATEGY...")
    
    # Profile Orders Timeline
    orders_df = pd.read_csv(os.path.join(RAW_DIR, target_files["orders"]))
    orders_date = pd.to_datetime(orders_df['order_date'], errors='coerce')
    print(f"  • Orders Timeline    -> Min: {orders_date.min()} | Max: {orders_date.max()}")
    
    # Profile Feedback Timeline
    fb_df = pd.read_csv(os.path.join(RAW_DIR, target_files["feedback"]))
    fb_date = pd.to_datetime(fb_df['feedback_date'], errors='coerce')
    print(f"  • Feedback Timeline  -> Min: {fb_date.min()} | Max: {fb_date.max()}")
    
    # Profile Marketing Timeline
    mkt_df = pd.read_csv(os.path.join(RAW_DIR, target_files["marketing"]))
    mkt_date = pd.to_datetime(mkt_df['date'], errors='coerce')
    print(f"  • Marketing Timeline -> Min: {mkt_date.min()} | Max: {mkt_date.max()}")

clean_diagnostics()

🔍 RUNNING CRITICAL SYSTEM DIAGNOSTICS & DATA SANITY VERIFICATION

📊 [1/3] VERIFYING RAW DISK LINE COUNTS (Checking for Truncation)...
  • qcommerce_orders.csv           -> Raw Lines on Disk: 5,001
  • qcommerce_order_items.csv      -> Raw Lines on Disk: 5,001
  • qcommerce_customers.csv        -> Raw Lines on Disk: 5,031
  • qcommerce_products.csv         -> Raw Lines on Disk: 269
  • qcommerce_inventory.csv        -> Raw Lines on Disk: 75,173
  • qcommerce_feedback.csv         -> Raw Lines on Disk: 5,001
  • qcommerce_marketing.csv        -> Raw Lines on Disk: 5,401

📈 Ratio Check: order_items to orders ratio is 1.00 items/order
  ✅ Pass: Items-to-orders ratio looks realistic.

🧩 [2/3] INVESTIGATING INVENTORY DATE PATTERNS (Mixed Format Check)...
  • Found Pattern Mapped Format: 'XX-XX-XXXX' -> Rows: 75,172
  • Actual Chronological Min Date: 2023-03-17 00:00:00
  • Actual Chronological Max Date: 2024-11-05 00:00:00

🗓️ [3/3] EXTRACTING DEFINITIVE TIMELINES FOR TIMELINE STRATEGY...
  •

In [7]:
import os
import pandas as pd

# Path configuration relative to the notebooks folder
INVENTORY_PATH = os.path.join("..", "data", "raw_source", "qcommerce_inventory.csv")

def audit_inventory_dates():
    if not os.path.exists(INVENTORY_PATH):
        print(f"[!] Target inventory file not found at: {INVENTORY_PATH}")
        return
        
    print("[*] Loading inventory records for date format validation...")
    df = pd.read_csv(INVENTORY_PATH)
    
    # Cast date to string and split by hyphen
    date_series = df['date'].astype(str)
    split_dates = date_series.str.split('-')
    
    # Ensure all splits successfully resulted in 3 parts (DD, MM, YYYY)
    valid_splits = split_dates[split_dates.str.len() == 3]
    
    # Extract structural components
    part1 = valid_splits.str[0].astype(int)
    part2 = valid_splits.str[1].astype(int)
    
    # An ambiguous row has BOTH part1 and part2 <= 12 (e.g., 05-06-2023)
    ambiguous_mask = (part1 <= 12) & (part2 <= 12)
    ambiguous_count = ambiguous_mask.sum()
    total_rows = len(df)
    
    print("=" * 60)
    print("🔍 INVENTORY DATE FORMAT DIAGNOSTIC REPORT")
    print("=" * 60)
    print(f"• Total Rows Scanned:          {total_rows:,}")
    print(f"• Structurally Ambiguous Rows: {ambiguous_count:,} ({(ambiguous_count/total_rows)*100:.2f}%)")
    
    if ambiguous_count > 0:
        print("\n📋 Sample of Ambiguous Rows:")
        print(df[ambiguous_mask].head(5).to_string(index=False))
        print("\n📋 Sample of Unambiguous Rows (Where day > 12):")
        unambiguous_mask = (part1 > 12) | (part2 > 12)
        print(df[unambiguous_mask].head(5).to_string(index=False))
    else:
        print("\n✅ Clean Pass: No ambiguous rows found.")

audit_inventory_dates()

[*] Loading inventory records for date format validation...
🔍 INVENTORY DATE FORMAT DIAGNOSTIC REPORT
• Total Rows Scanned:          75,172
• Structurally Ambiguous Rows: 29,399 (39.11%)

📋 Sample of Ambiguous Rows:
 product_id       date  stock_received  damaged_stock
      11422 01-04-2023               3              2
     848226 01-04-2023               0              2
     890623 01-04-2023               4              0
     222892 01-04-2023               0              2
     818990 01-04-2023               3              2

📋 Sample of Unambiguous Rows (Where day > 12):
 product_id       date  stock_received  damaged_stock
     153019 17-03-2023               4              2
     848226 17-03-2023               4              2
     965755 17-03-2023               1              0
      39154 17-03-2023               4              0
      34186 17-03-2023               3              2


In [8]:
import os
import pandas as pd

# Path configuration relative to the notebooks folder
RAW_DIR = os.path.join("..", "data", "raw_source")

target_files = {
    "orders": "qcommerce_orders.csv",
    "items": "qcommerce_order_items.csv",
    "customers": "qcommerce_customers.csv",
    "products": "qcommerce_products.csv",
    "inventory": "qcommerce_inventory.csv",
    "feedback": "qcommerce_feedback.csv",
    "marketing": "qcommerce_marketing.csv"
}

def run_full_data_quality_report():
    dfs = {}
    
    print("=" * 70)
    print("🔍 STAGE 1: LOADING DATASETS & ANALYZING PROFILE METRICS")
    print("=" * 70)
    
    for nickname, filename in target_files.items():
        full_path = os.path.join(RAW_DIR, filename)
        if not os.path.exists(full_path):
            print(f"[!] Critical Error: Missing required file -> {filename}")
            return
        
        # Load full dataframe
        df = pd.read_csv(full_path)
        dfs[nickname] = df
        
        total_rows = len(df)
        null_counts = df.isnull().sum()
        cols_with_nulls = null_counts[null_counts > 0]
        duplicate_count = df.duplicated().sum()
        
        print(f"\n📊 Dataset: {filename}")
        print(f"  • Total Records:  {total_rows:,}")
        print(f"  • Exact Duplicates: {duplicate_count:,} ({(duplicate_count/total_rows)*100:.2f}%)")
        
        if len(cols_with_nulls) > 0:
            print("  • Missing Values Detected:")
            for col, count in cols_with_nulls.items():
                print(f"    - {col}: {count:,} missing rows ({(count/total_rows)*100:.2f}%)")
        else:
            print("  • Missing Values: Clean (0 nulls found)")
            
    print("\n" + "=" * 70)
    print("🧩 STAGE 2: REFERENTIAL INTEGRITY & FOREIGN KEY AUDIT")
    print("=" * 70)
    
    # 1. Check Items -> Orders link
    orphaned_items = dfs['items'][~dfs['items']['order_id'].isin(dfs['orders']['order_id'])]
    print(f"\n🔗 Checking Order Items -> Master Orders linkage:")
    print(f"  • Total line items scanned: {len(dfs['items']):,}")
    print(f"  • Orphaned item rows (No matching order_id): {len(orphaned_items):,}")

    # 2. Check Items -> Products link
    orphaned_products = dfs['items'][~dfs['items']['product_id'].isin(dfs['products']['product_id'])]
    print(f"\n🔗 Checking Order Items -> Products Catalog linkage:")
    print(f"  • Orphaned item rows (No matching product_id): {len(orphaned_products):,}")

    # 3. Check Orders -> Customers link
    orphaned_customers = dfs['orders'][~dfs['orders']['customer_id'].isin(dfs['customers']['customer_id'])]
    print(f"\n🔗 Checking Master Orders -> Customer Profiles linkage:")
    print(f"  • Total orders scanned: {len(dfs['orders']):,}")
    print(f"  • Orphaned order rows (No matching customer_id): {len(orphaned_customers):,}")

    # 4. Check Feedback -> Orders link
    orphaned_feedback = dfs['feedback'][~dfs['feedback']['order_id'].isin(dfs['orders']['order_id'])]
    print(f"\n🔗 Checking Customer Feedback -> Master Orders linkage:")
    print(f"  • Total reviews scanned: {len(dfs['feedback']):,}")
    print(f"  • Orphaned review rows (No matching order_id): {len(orphaned_feedback):,}")

    # 5. Check Inventory -> Products link
    orphaned_inv_products = dfs['inventory'][~dfs['inventory']['product_id'].isin(dfs['products']['product_id'])]
    print(f"\n🔗 Checking Inventory Records -> Products Catalog linkage:")
    print(f"  • Total inventory lines scanned: {len(dfs['inventory']):,}")
    print(f"  • Orphaned inventory rows (No matching product_id): {len(orphaned_inv_products):,}")
    print("\n" + "=" * 70)

run_full_data_quality_report()

🔍 STAGE 1: LOADING DATASETS & ANALYZING PROFILE METRICS

📊 Dataset: qcommerce_orders.csv
  • Total Records:  5,000
  • Exact Duplicates: 0 (0.00%)
  • Missing Values: Clean (0 nulls found)

📊 Dataset: qcommerce_order_items.csv
  • Total Records:  5,000
  • Exact Duplicates: 0 (0.00%)
  • Missing Values: Clean (0 nulls found)

📊 Dataset: qcommerce_customers.csv
  • Total Records:  2,500
  • Exact Duplicates: 0 (0.00%)
  • Missing Values: Clean (0 nulls found)

📊 Dataset: qcommerce_products.csv
  • Total Records:  268
  • Exact Duplicates: 0 (0.00%)
  • Missing Values: Clean (0 nulls found)

📊 Dataset: qcommerce_inventory.csv
  • Total Records:  75,172
  • Exact Duplicates: 0 (0.00%)
  • Missing Values: Clean (0 nulls found)

📊 Dataset: qcommerce_feedback.csv
  • Total Records:  5,000
  • Exact Duplicates: 0 (0.00%)
  • Missing Values: Clean (0 nulls found)

📊 Dataset: qcommerce_marketing.csv
  • Total Records:  5,400
  • Exact Duplicates: 0 (0.00%)
  • Missing Values: Clean (0 nulls fou

In [10]:
import os
import pandas as pd

# Path configuration relative to the notebooks folder
RAW_DIR = os.path.join("..", "data", "raw_source")
ORDERS_PATH = os.path.join(RAW_DIR, "qcommerce_orders.csv")

def audit_delivery_metrics_mismatch():
    if not os.path.exists(ORDERS_PATH):
        print(f"Error: Target orders file not found at {ORDERS_PATH}")
        return

    # Load and explicitly parse timestamps
    df = pd.read_csv(ORDERS_PATH)
    df['order_date'] = pd.to_datetime(df['order_date'], format="%Y-%m-%d %H:%M:%S")
    df['promised_delivery_time'] = pd.to_datetime(df['promised_delivery_time'], format="%Y-%m-%d %H:%M:%S")
    df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'], format="%Y-%m-%d %H:%M:%S")

    # Compute explicit boolean late flag based strictly on timestamps
    df['computed_late'] = df['actual_delivery_time'] > df['promised_delivery_time']

    # Generate Cross-Tabulation Matrix
    cross_tab = pd.crosstab(df['delivery_status'], df['computed_late'], margins=True, margins_name="Total")
    
    print("=" * 70)
    print("🔍 CROSS-TABULATION: RAW STATUS LABEL VS COMPUTED TIMESTAMP LATE")
    print("=" * 70)
    print(cross_tab)
    print("\n" + "=" * 70)

    # Isolate conflicting logic signatures
    ontime_but_late_ts = df[(df['delivery_status'] == 'On Time') & (df['computed_late'] == True)]
    delayed_but_early_ts = df[(df['delivery_status'].isin(['Slightly Delayed', 'Significantly Delayed'])) & (df['computed_late'] == False)]

    if len(ontime_but_late_ts) > 0:
        print(f"⚠️ Critical Anomaly: {len(ontime_but_late_ts)} rows are labeled 'On Time' but timestamps show Actual > Promised.")
        print("\n📋 Sample Conflict Rows ('On Time' Label but late timestamps):")
        print(ontime_but_late_ts[['order_id', 'order_date', 'promised_delivery_time', 'actual_delivery_time', 'delivery_status']].head(5).to_string(index=False))
        print("\n" + "=" * 70)

    if len(delayed_but_early_ts) > 0:
        print(f"⚠️ Critical Anomaly: {len(delayed_but_early_ts)} rows are labeled 'Delayed' but timestamps show Actual <= Promised.")
        print("\n📋 Sample Conflict Rows ('Delayed' Label but early/on-time timestamps):")
        print(delayed_but_early_ts[['order_id', 'order_date', 'promised_delivery_time', 'actual_delivery_time', 'delivery_status']].head(5).to_string(index=False))
        print("\n" + "=" * 70)

audit_delivery_metrics_mismatch()

🔍 CROSS-TABULATION: RAW STATUS LABEL VS COMPUTED TIMESTAMP LATE
computed_late          False  True  Total
delivery_status                          
On Time                 1902  1568   3470
Significantly Delayed      0   493    493
Slightly Delayed           0  1037   1037
Total                   1902  3098   5000

⚠️ Critical Anomaly: 1568 rows are labeled 'On Time' but timestamps show Actual > Promised.

📋 Sample Conflict Rows ('On Time' Label but late timestamps):
  order_id          order_date promised_delivery_time actual_delivery_time delivery_status
1549769649 2024-05-28 13:14:29    2024-05-28 13:25:29  2024-05-28 13:27:29         On Time
9185164487 2024-09-23 13:07:12    2024-09-23 13:25:12  2024-09-23 13:29:12         On Time
5427684290 2023-11-20 05:00:39    2023-11-20 05:17:39  2023-11-20 05:18:39         On Time
4898355547 2023-04-16 18:50:37    2023-04-16 19:01:37  2023-04-16 19:02:37         On Time
6568151549 2024-03-31 06:26:48    2024-03-31 06:37:48  2024-03-31 06:39:4

In [11]:
import os
import pandas as pd

RAW_DIR = os.path.join("..", "data", "raw_source")
ORDERS_PATH = os.path.join(RAW_DIR, "qcommerce_orders.csv")

def find_grace_period_window():
    df = pd.read_csv(ORDERS_PATH)
    df['promised_delivery_time'] = pd.to_datetime(df['promised_delivery_time'], format="%Y-%m-%d %H:%M:%S")
    df['actual_delivery_time'] = pd.to_datetime(df['actual_delivery_time'], format="%Y-%m-%d %H:%M:%S")
    
    # Isolate rows where the label says "On Time" but it was mathematically past the deadline
    grace_window_df = df[
        (df['delivery_status'] == 'On Time') & 
        (df['actual_delivery_time'] > df['promised_delivery_time'])
    ].copy()
    
    # Calculate delivery delay in minutes
    delay_minutes = (grace_window_df['actual_delivery_time'] - grace_window_df['promised_delivery_time']).dt.total_seconds() / 60.0
    
    print("=" * 60)
    print("📋 GRACE PERIOD MARGIN METRICS")
    print("=" * 60)
    print(f"• Minimum delay in group: {delay_minutes.min():.2f} minutes")
    print(f"• Maximum delay in group: {delay_minutes.max():.2f} minutes")
    print(f"• Most common delay value: {delay_minutes.mode().values[0]:.2f} minutes")
    print("=" * 60)

find_grace_period_window()

📋 GRACE PERIOD MARGIN METRICS
• Minimum delay in group: 1.00 minutes
• Maximum delay in group: 5.00 minutes
• Most common delay value: 5.00 minutes


***AFTER CLEANING AND TRANSFORMATION***

In [9]:
import os
import pandas as pd
import numpy as np

# Path configurations relative to notebooks folder
FACTS_DIR = os.path.join("..", "data", "processed_facts")
FACT_ORDERS_PATH = os.path.join(FACTS_DIR, "fact_order_line.csv")

def calibrate_alert_thresholds():
    if not os.path.exists(FACT_ORDERS_PATH):
        print(f"Error: Target fact table not found at {FACT_ORDERS_PATH}. Run build_facts.py first.")
        return

    # 1. Load compiled transactional fact layer
    df = pd.read_csv(FACT_ORDERS_PATH)
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['date_only'] = df['order_date'].dt.date

    # 2. Compute true operational metrics per unique day
    print("Extracting daily timeline metrics for statistical calibration...")
    
    daily_stats = df.groupby('date_only').agg(
        daily_order_volume=('order_id', 'nunique'),
        daily_revenue=('order_total', 'sum'),
        total_late_deliveries=('delivery_delay_minutes', lambda x: (x > 0).sum())
    ).reset_index()

    # Calculate precise daily SLA failure rate percentage
    daily_stats['sla_failure_rate_pct'] = (daily_stats['total_late_deliveries'] / daily_stats['daily_order_volume']) * 100.0

    # 3. Analyze Statistical Distributions & Percentiles
    print("\n" + "="*50)
    print("📈 DAILY OPERATIONAL METRICS DISTRIBUTION")
    print("="*50)
    
    metrics = {
        "Daily Order Volume (Units)": daily_stats['daily_order_volume'],
        "Daily Revenue (INR)": daily_stats['daily_revenue'],
        "Daily SLA Failure Rate (%)": daily_stats['sla_failure_rate_pct']
    }

    for label, series in metrics.items():
        print(f"\n📊 Metric: {label}")
        print(f"  • Mean (Average):     {series.mean():.2f}")
        print(f"  • Median (50th Pctl): {series.median():.2f}")
        print(f"  • Minimum Observed:   {series.min():.2f}")
        print(f"  • Maximum Observed:   {series.max():.2f}")
        print(f"  • 10th Percentile:    {series.quantile(0.10):.2f} (Low Bound)")
        print(f"  • 90th Percentile:    {series.quantile(0.90):.2f} (High Bound)")

    # 4. Calibration Recommendations Rules
    print("\n" + "="*50)
    print("⚙️ RECOMMENDED ALERT THRESHOLD CONFIGURATIONS")
    print("="*50)
    print(f"🚨 SLA Breach Trigger: Set to > {daily_stats['sla_failure_rate_pct'].quantile(0.90):.1f}%")
    print("   (Triggers when daily late deliveries exceed the 90th percentile of worst operational days)")
    
    print(f"\n📉 Revenue Drop Trigger: Set to < {daily_stats['daily_revenue'].quantile(0.10):.2f} INR")
    print("   (Triggers an anomaly alert when sales plummet into the bottom 10th percentile of performance)")

calibrate_alert_thresholds()

Extracting daily timeline metrics for statistical calibration...

📈 DAILY OPERATIONAL METRICS DISTRIBUTION

📊 Metric: Daily Order Volume (Units)
  • Mean (Average):     8.33
  • Median (50th Pctl): 8.00
  • Minimum Observed:   1.00
  • Maximum Observed:   17.00
  • 10th Percentile:    4.00 (Low Bound)
  • 90th Percentile:    13.00 (High Bound)

📊 Metric: Daily Revenue (INR)
  • Mean (Average):     18348.85
  • Median (50th Pctl): 17453.18
  • Minimum Observed:   935.64
  • Maximum Observed:   46926.32
  • 10th Percentile:    9049.13 (Low Bound)
  • 90th Percentile:    28714.00 (High Bound)

📊 Metric: Daily SLA Failure Rate (%)
  • Mean (Average):     61.90
  • Median (50th Pctl): 62.50
  • Minimum Observed:   0.00
  • Maximum Observed:   100.00
  • 10th Percentile:    40.00 (Low Bound)
  • 90th Percentile:    85.71 (High Bound)

⚙️ RECOMMENDED ALERT THRESHOLD CONFIGURATIONS
🚨 SLA Breach Trigger: Set to > 85.7%
   (Triggers when daily late deliveries exceed the 90th percentile of worst 

In [13]:
import os
import pandas as pd

FACT_ORDERS_PATH = os.path.join("..", "data", "processed_facts", "fact_order_line.csv")

def calibrate_corrected_thresholds():
    if not os.path.exists(FACT_ORDERS_PATH):
        print("Error: Rebuild your fact table first.")
        return

    df = pd.read_csv(FACT_ORDERS_PATH)
    df['order_date'] = pd.to_datetime(df['order_date'])
    df['date_only'] = df['order_date'].dt.date

    # Aggregate based strictly on corporate SLA boundaries
    daily_stats = df.groupby('date_only').agg(
        daily_order_volume=('order_id', 'nunique'),
        daily_revenue=('order_total', 'sum'),
        total_late_orders=('is_late', 'sum')
    ).reset_index()

    daily_stats['sla_failure_rate_pct'] = (daily_stats['total_late_orders'] / daily_stats['daily_order_volume']) * 100.0

    print("="*60)
    print("📈 RE-CALIBRATED DAILY METRICS DISTRIBUTION")
    print("="*60)
    
    metrics = {
        "Daily Order Volume": daily_stats['daily_order_volume'],
        "Daily Revenue (INR)": daily_stats['daily_revenue'],
        "Daily SLA Failure Rate (%)": daily_stats['sla_failure_rate_pct']
    }

    for label, series in metrics.items():
        print(f"\n📊 Metric: {label}")
        print(f"  • Mean:               {series.mean():.2f}")
        print(f"  • Median (50th Pctl): {series.median():.2f}")
        print(f"  • 10th Percentile:    {series.quantile(0.10):.2f}")
        print(f"  • 90th Percentile:    {series.quantile(0.90):.2f}")

    print("\n" + "="*60)
    print("⚙️ CORRECTED EXTERNAL CONFIGURATION VARIABLE BOUNDS")
    print("="*60)
    print(f"• SLA_BREACH_THRESHOLD_PCT: {daily_stats['sla_failure_rate_pct'].quantile(0.90):.1f}")
    print(f"• REVENUE_DROP_THRESHOLD_INR: {daily_stats['daily_revenue'].quantile(0.10):.2f}")

calibrate_alert_thresholds = calibrate_corrected_thresholds
calibrate_alert_thresholds()

📈 RE-CALIBRATED DAILY METRICS DISTRIBUTION

📊 Metric: Daily Order Volume
  • Mean:               8.33
  • Median (50th Pctl): 8.00
  • 10th Percentile:    4.00
  • 90th Percentile:    13.00

📊 Metric: Daily Revenue (INR)
  • Mean:               18348.85
  • Median (50th Pctl): 17453.18
  • 10th Percentile:    9049.13
  • 90th Percentile:    28714.00

📊 Metric: Daily SLA Failure Rate (%)
  • Mean:               30.31
  • Median (50th Pctl): 28.57
  • 10th Percentile:    10.00
  • 90th Percentile:    50.00

⚙️ CORRECTED EXTERNAL CONFIGURATION VARIABLE BOUNDS
• SLA_BREACH_THRESHOLD_PCT: 50.0
• REVENUE_DROP_THRESHOLD_INR: 9049.13


In [14]:
import os
import sys
import pandas as pd

# Step 1: Map execution paths back to your active logic scripts
sys.path.append(os.path.abspath(os.path.join("..")))

from scripts.compute_metrics import calculate_daily_slice_metrics
from scripts.check_thresholds import evaluate_metrics_against_thresholds
from scripts.send_alert import dispatch_anomaly_alert

REPLAY_DIR = os.path.join("..", "data", "daily_replay")

def execute_manual_replay_simulation(sample_days_count: int = 3):
    """
    Simulates sequential daily ingestion, transformation, and SLA/revenue evaluation
    loops directly from the disk-partitioned history folders.
    """
    if not os.path.exists(REPLAY_DIR):
        print(f"[!] Target replay directory not found at: {REPLAY_DIR}. Run chunk_daily.py first.")
        return
        
    # Isolate partition strings chronologically
    partitions = sorted([d for d in os.listdir(REPLAY_DIR) if d.startswith("date_partition_")])
    test_subset = partitions[:sample_days_count]
    
    print(f"[*] Starting end-to-end replay simulation for {len(test_subset)} operational cycles...")
    
    for partition_name in test_subset:
        # Extract operational target date from folder name string
        target_date_str = partition_name.replace("date_partition_", "")
        partition_path = os.path.join(REPLAY_DIR, partition_name)
        
        print("\n" + "="*60)
        print(f"📅 PROCESSING OPERATIONAL CYCLE DATE: {target_date_str}")
        print("="*60)
        
        # Load incremental files mimicry
        orders_df = pd.read_csv(os.path.join(partition_path, "orders.csv"))
        items_df = pd.read_csv(os.path.join(partition_path, "order_items.csv"))
        
        print(f"  • Extracted daily slice: {len(orders_df)} order logs loaded.")
        
        # --- Transformation Simulation Phase ---
        # Apply strict type overrides and generate the is_late business flag mapping
        orders_df["order_id"] = orders_df["order_id"].astype(int)
        items_df["order_id"] = items_df["order_id"].astype(int)
        
        day_slice = pd.merge(items_df, orders_df, on="order_id", how="inner")
        day_slice["is_late"] = (day_slice["delivery_status"] != "On Time").astype(int)
        
        # --- Metrics Calculation Phase ---
        daily_metrics = calculate_daily_slice_metrics(day_slice)
        print(f"  • Computed Metrics -> Orders: {daily_metrics['order_count']} | Revenue: INR {daily_metrics['total_revenue']:,} | SLA Failure: {daily_metrics['sla_failure_pct']}%")
        
        # --- Threshold Evaluation Phase ---
        detected_anomalies = evaluate_metrics_against_thresholds(daily_metrics)
        
        if detected_anomalies:
            print(f"  • ⚠️ Threshold Breach Triggered! Found {len(detected_anomalies)} operational boundary exceptions.")
            for anomaly in detected_anomalies:
                print(f"    - Alert: [{anomaly['metric']}] {anomaly['actual_value']} {anomaly['condition']} (Limit: {anomaly['threshold_value']})")
                
            # --- Email Dispatch Hook Phase ---
            # This calls your secure environment variable hook setup to run validation checks
            dispatch_anomaly_alert(detected_anomalies, target_date_str)
        else:
            print("  • ✅ Operational limits stable. Zero anomalies logged for this date window.")

if __name__ == "__main__":
    execute_manual_replay_simulation(sample_days_count=3)

[*] Starting end-to-end replay simulation for 3 operational cycles...

📅 PROCESSING OPERATIONAL CYCLE DATE: 2023-03-16
  • Extracted daily slice: 9 order logs loaded.
  • Computed Metrics -> Orders: 9 | Revenue: INR 16,066.04 | SLA Failure: 33.33%
  • ✅ Operational limits stable. Zero anomalies logged for this date window.

📅 PROCESSING OPERATIONAL CYCLE DATE: 2023-03-17
  • Extracted daily slice: 6 order logs loaded.
  • Computed Metrics -> Orders: 6 | Revenue: INR 10,818.65 | SLA Failure: 16.67%
  • ✅ Operational limits stable. Zero anomalies logged for this date window.

📅 PROCESSING OPERATIONAL CYCLE DATE: 2023-03-18
  • Extracted daily slice: 7 order logs loaded.
  • Computed Metrics -> Orders: 7 | Revenue: INR 18,575.53 | SLA Failure: 14.29%
  • ✅ Operational limits stable. Zero anomalies logged for this date window.


In [18]:
import os
import sys

# Append path back to root to pull dependencies
sys.path.append(os.path.abspath(os.path.join("..")))
from scripts.send_alert import dispatch_anomaly_alert

# Create an intentional critical failure payload
mock_forced_breaches = [
    {
        "metric": "Daily Revenue",
        "actual_value": "INR 4,250.00",
        "threshold_value": "INR 9,049.13",
        "condition": "Fell Below"
    },
    {
        "metric": "SLA Failure Rate",
        "actual_value": "66.67%",
        "threshold_value": "50.0%",
        "condition": "Exceeded"
    }
]

print("[*] Initiating forced alert transmission test...")
success = dispatch_anomaly_alert(mock_forced_breaches, "TEST-CYCLE-LOG")

if success:
    print("✅ Success! Check the inbox of the receiver email address defined in your .env file.")
else:
    print("❌ Failed. Check your terminal output logs above for connection or authentication errors.")

[*] Initiating forced alert transmission test...
[!] Critical SMTP Transmission Failure: (534, b'5.7.9 Application-specific password required. For more information, go to\n5.7.9  https://support.google.com/mail/?p=InvalidSecondFactor 5a478bee46e88-30f290b6bc2sm10557916eec.27 - gsmtp')
❌ Failed. Check your terminal output logs above for connection or authentication errors.
